## 1. 數據分析Agent介紹

這是一個零售交易數據分析的AI Agent (已經寫在`utils.py`)，使用者只需要利用自然語言對話，就可以得到想要的資料，並且請Agent分析資料得到一些業務上的想法，若有需要的話還可以請Agent畫出合適的圖表。

<img src="images/Agent-flow.png" width="700"/>
<br>

這個Agent的組成包含三個工具，會依據使用者的需求，決定需要調用什麼工具：
1. 工具1: 資料查詢工具 (Look Up Sales Data Tool)
    - 連接數據庫執行查詢
    - 提取銷售、產品、店舖等基礎數據
    - 處理結構化數據檢索需求
2. 工具2: 資料分析工具 (Data Analysis Tool)
    - 執行統計計算與數據分析
    - 提供業務指標計算
    - 生成分析報告與洞察
3. 工具3: 資料視覺化工具 (Data Visualization Tool)
    - 自動生成各類圖表
    - 支援長條圖、圓餅圖、趨勢圖等
    - 將數據轉化為直觀的視覺呈現


目前已經內建了一個零售交易的資料表(`./data/Store_Sales_Price_Elasticity_Promotions_Data.parquet`)：
 | 欄位名稱               | 中文說明   | 資料類型 | 範例值                 |
  |--------------------|--------|------|---------------------|
  | Store_Number       | 門市編號   | 數值   | 1320, 2310, 3080    |
  | SKU_Coded          | 商品編碼   | 數值   | 6172800             |
  | Product_Class_Code | 商品類別代碼 | 數值   | 22875               |
  | Sold_Date          | 銷售日期   | 日期   | 2021-11-02          |
  | Qty_Sold           | 銷售數量   | 數值   | 1, 2, 3             |
  | Total_Sale_Value   | 銷售總金額  | 金額   | 18.95, 37.90, 56.85 |
  | On_Promo           | 是否促銷   | 布林值  | 0 (否), 1 (是)        |

## 2. Agent評估總覽

why評估?
- 定位問題根源：當 Agent 表現不佳時，能快速找到是哪個環節出問題
- 漸進式改進：可以針對性地改善特定的節點
- 評估驅動開發 (Evaluation-Driven Development)


### (1) AI Agent 評估的面向 (包含但不限於)
```
    ┌─────────────────────┐
    │   Router評估         │  ← 決策能力
    │   (工具選擇)         │
    └─────────────────────┘

    ┌─────────────────────┐
    │   工具性能評估       │  ← 個別工具品質
    │   (SQL/分析/視覺化)  │
    └─────────────────────┘

    ┌─────────────────────┐
    │   端到端評估         │  ← 使用者體驗
    │   (整體效果)         │
    └─────────────────────┘
```
### (2) 計算分數的幾種方式

- Code-Based Evals: 有一個很明確的評估標準 (例如: 跟ground truth比對、是否有包含某些關鍵字...等等)，直接用程式去評估，此種方法每一次執行評估結果都相同。

- LLM-as-a-Judge Evals: 使用LLM去評估AI agent的表現，此種方法每一次執行評估結果有可能會不同。

- Human Annotations:人工判斷: 人工標註或者是使用者回饋，此種方法每一次執行評估結果有可能會不同。

<img src="images/eval-methods.png" width="500"/>


# 3. Agent資料紀錄 (Tracing)

為了要方便記錄AI Agent內部的歷程，以利後續評估Agent成效，這裡會使用Phoenix作為tracing的平台，並且利用phoenix的套件進行評估。

Phoenix設定：

step0. Phoenix (https://app.phoenix.arize.com/)

step1. 在Phoenix上面建立space，你的PHOENIX_COLLECTOR_ENDPOINT為 `https://app.phoenix.arize.com/s/<space_name>`，以圖中為例，PHOENIX_COLLECTOR_ENDPOINT為 `https://app.phoenix.arize.com/s/tylin-amice`<br>
<img src="images/phoenix-space.png" width="500"/>


step2. 點左欄下方的Profile，建立phoenix API Key <br><img src="images/phoenix-api.png" width="500"/>

# 4. 程式範例

在這個範例中，我們想對Agent的Router執行的正確性進行評估 (LLM-as-a-judge)

1. 先進行環境設定，讓agent執行的過程連接到phoenix平台
2. 請agent執行一些任務，產生我們要評估的資料
3. 從phoenix平台撈agent執行的歷程
4. 進行評估

References: https://learn.deeplearning.ai/courses/evaluating-ai-agents

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import phoenix as px
import os
import json
from tqdm import tqdm
from phoenix.evals import (
    TOOL_CALLING_PROMPT_TEMPLATE, 
    llm_classify,
    OpenAIModel
)
from phoenix.trace import SpanEvaluations
from phoenix.trace.dsl import SpanQuery
from openinference.instrumentation import suppress_tracing

import nest_asyncio
nest_asyncio.apply()

## (1) 環境設定

In [ ]:
# 記得先在.env中設定OPENAI_API_KEY和PHOENIX_API_KEY
# (phoenix設定見tracing介紹段落)

from dotenv import load_dotenv
load_dotenv()
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com/s/<space>" # 記得換成你自己的


In [4]:
from utils import run_agent, start_main_span, tools, get_phoenix_endpoint, PROJECT_NAME

Attempting to instrument while already instrumented


OpenTelemetry Tracing Details
|  Phoenix Project: evaluating-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/s/tylin-amice/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



## 2. 請AI Agent執行一些任務


In [ ]:
agent_questions = [
    "所有門市的總營收是多少?",
    "最受歡迎的商品 SKU 是哪一個?",
    "哪一家門市的銷售量最高?",
    "請製作一張柱狀圖，顯示各門市的總銷售額",
    "有多少百分比的商品是在促銷期間售出的",
    "平均每筆交易金額是多少?"
]

for question in tqdm(agent_questions, desc="Processing questions"):
    try:
        ret = start_main_span([{"role": "user", "content": question}])
    except Exception as e:
        print(f"Error processing question: {question}")
        print(e)
        continue

## 3. 從平台上將相關的紀錄抓下來

In [6]:
query = SpanQuery().where(
    # Filter for the `LLM` span kind.
    # The filter condition is a string of valid Python boolean expression.
    "span_kind == 'LLM'",
).select(
    question="input.value",
    tool_call="llm.tools",
    model="llm.model_name"
)

# The Phoenix Client can take this query and return the dataframe.
tool_calls_df = px.Client().query_spans(query, 
                                        project_name=PROJECT_NAME, 
                                        timeout=None)
tool_calls_df = tool_calls_df.dropna(subset=["tool_call"])

tool_calls_df.head()

,question,tool_call,model
context.span_id,,,
5bca82eac6c01283,"{""messages"": [{""role"": ""user"", ""content"": ""請製作...","[{'tool': {'json_schema': '{""type"": ""function""...",gpt-3.5-turbo-0125
c17276facadc733b,"{""messages"": [{""role"": ""user"", ""content"": ""有多少...","[{'tool': {'json_schema': '{""type"": ""function""...",gpt-3.5-turbo-0125
835512de87c88056,"{""messages"": [{""role"": ""user"", ""content"": ""Wha...","[{'tool': {'json_schema': '{""type"": ""function""...",gpt-4o-mini-2024-07-18
fe9609257d18a932,"{""messages"": [{""role"": ""user"", ""content"": ""Wha...","[{'tool': {'json_schema': '{""type"": ""function""...",gpt-4o-mini-2024-07-18
4f72386a6758242f,"{""messages"": [{""role"": ""user"", ""content"": ""Wha...","[{'tool': {'json_schema': '{""type"": ""function""...",gpt-4o-mini-2024-07-18


## 4. 對AI agent進行評估

使用phoenix套件中的prompt，也可以依據自己的情境客製化

In [7]:
print(TOOL_CALLING_PROMPT_TEMPLATE)


You are an evaluation assistant evaluating questions and tool calls to
determine whether the tool called would answer the question. The tool
calls have been generated by a separate agent, and chosen from the list of
tools provided below. It is your job to decide whether that agent chose
the right tool to call.

    [BEGIN DATA]
    ************
    [Question]: {question}
    ************
    [Tool Called]: {tool_call}
    [END DATA]

Your response must be single word, either "correct" or "incorrect",
and should not contain any text or characters aside from that word.
"incorrect" means that the chosen tool would not answer the question,
the tool includes information that is not presented in the question,
or that the tool signature includes parameter values that don't match
the formats specified in the tool signatures below.

"correct" means the correct tool call was chosen, the correct parameters
were extracted from the question, the tool call generated is runnable and correct,
and tha

In [8]:
with suppress_tracing():
    # llm as judge
    tool_call_eval = llm_classify(
        dataframe = tool_calls_df,
        template = TOOL_CALLING_PROMPT_TEMPLATE.template[0].template.replace("{tool_definitions}", 
                                                                 json.dumps(tools).replace("{", '"').replace("}", '"')),
        rails = ['correct', 'incorrect'],
        model=OpenAIModel(model="gpt-4o-mini"),
        provide_explanation=True
    )

tool_call_eval['score'] = tool_call_eval.apply(lambda x: 1 if x['label']=='correct' else 0, axis=1)

tool_call_eval.head()

llm_classify |          | 0/73 (0.0%) | ⏳ 00:00<? | ?it/s

,label,explanation,exceptions,execution_status,execution_seconds,score
context.span_id,,,,,,
5bca82eac6c01283,incorrect,"The tool call made was to 'lookup_sales_data',...",[],COMPLETED,2.327628,0
c17276facadc733b,correct,"The tool call made was to 'lookup_sales_data',...",[],COMPLETED,1.872854,1
835512de87c88056,correct,The tool call to 'lookup_sales_data' is approp...,[],COMPLETED,1.841123,1
fe9609257d18a932,correct,The chosen tools include a lookup function to ...,[],COMPLETED,2.173995,1
4f72386a6758242f,correct,The chosen tools include 'lookup_sales_data' t...,[],COMPLETED,1.815414,1


In [9]:
# 看看不同模型的分數差異
tool_call_eval.join(tool_calls_df, on="context.span_id").groupby("model")["score"].mean()

model
gpt-3.5-turbo-0125        0.914894
gpt-4o-mini-2024-07-18    0.807692
Name: score, dtype: float64

# 5. Assignments
記得要將data資料夾以及utils.py下載下來

### Q1. 評估工具2 (Data Analysis Tool)
請評估資料分析工具生成的分析，以「分析是否清楚」作為評估的指標，判斷分析結果是否夠精確、有條理，並且直接回應分析的問題，沒有多餘或不必要的結果產生。

Hint: 自行設計prompt，利用LLM-as-a-judge的方式進行資料分析工具表現的評估。

In [10]:
CLARITY_LLM_JUDGE_PROMPT = """
在這項任務中，你將會看到一個「問題（query）」與其對應的「回答（answer）」。你的目標是評估這個回答在回應問題時的清楚程度。
一個清楚的回答應該是精確、連貫，並且直接回應問題，不應加入不必要的複雜內容或含糊之處。
一個不清楚的回答則是模糊、結構混亂或難以理解的，即使它在事實上可能正確。

你的回答應該是一個單字：「clear」或者是「unclear」，而且不得包含其他文字或符號。
「clear」表示這個回答結構良好、易於理解，而且有恰當地回應問題。
「unclear」表示這個回答的某些部分應該可以有更好的結構或表達方式。

請你仔細閱讀問題與回答，然後判斷它們的清楚程度。

在分析問題與回答之後，你必須寫出詳細的解釋，說明你為什麼選擇「clear」或「unclear」。

[BEGIN DATA]
Query: {query}
Answer: {response}
[END DATA]

在說明中不要一開始就揭示你的最後判斷。
你的解釋應該包含具體的觀察，例如：這個回答是否結構良好、是否直接回應了問題、是否有多餘或令人困惑的內容等。
標註: "clear" or "unclear"
"""

In [11]:
query = SpanQuery().where(
    "span_kind=='AGENT'"
).select(
    response="output.value",
    query="input.value"
)

# The Phoenix Client can take this query and return the dataframe.
clarity_df = px.Client().query_spans(query, 
                                     project_name=PROJECT_NAME,
                                     timeout=None).fillna("")

clarity_df.head()

,response,query
context.span_id,,
e071da59a81dd7d1,"所有門市的總營收是 13,272,640 元。","[{""role"": ""user"", ""content"": ""所有門市的總營收是多少?""}]"
1c564d9d9fccd54d,"The most popular product SKU was **6200700**, ...","[{""role"": ""user"", ""content"": ""What was the mos..."
6919a00c2e9a74c9,The percentage of items sold on promotion is 0...,"[{""role"": ""user"", ""content"": ""What percentage ..."
ab1c39cf9adc52e1,The total revenue across all stores was approx...,"[{""role"": ""user"", ""content"": ""What was the tot..."
9a4703cc970dfb32,The store with the highest sales volume is Sto...,"[{""role"": ""user"", ""content"": ""Which store had ..."


In [12]:
with suppress_tracing():
    clarity_eval = llm_classify(
        dataframe = clarity_df,
        template = CLARITY_LLM_JUDGE_PROMPT,
        rails = ['clear', 'unclear'],
        model=OpenAIModel(model="gpt-4o-mini"),
        provide_explanation=True
    )

clarity_eval['score'] = clarity_eval.apply(lambda x: 1 if x['label']=='clear' else 0, axis=1)

clarity_eval.head()

llm_classify |          | 0/36 (0.0%) | ⏳ 00:00<? | ?it/s

,label,explanation,exceptions,execution_status,execution_seconds,score
context.span_id,,,,,,
e071da59a81dd7d1,clear,這個回答直接回應了問題，清楚地提供了所有門市的總營收數字，並且沒有包含多餘或令人困惑的內容。...,[],COMPLETED,1.825474,1
1c564d9d9fccd54d,clear,The answer directly addresses the question by ...,[],COMPLETED,1.946490,1
6919a00c2e9a74c9,clear,The answer directly addresses the question by ...,[],COMPLETED,1.828475,1
ab1c39cf9adc52e1,clear,The answer directly addresses the user's quest...,[],COMPLETED,1.693367,1
9a4703cc970dfb32,clear,The answer directly addresses the question by ...,[],COMPLETED,1.741797,1


### Q2. 評估工具3 (Data Visualization Tool)

請評估資料視覺化的工具的能力，根據這個Tool產生的程式碼，以「是否可成功執行」作為評估的指標。

Hint: 自行設計涵式，確認產生出的程式是否可執行，用Code-Based Eval的方式給予評估分數。

In [13]:
query = SpanQuery().where(
    "name =='generate_visualization'"
).select(
    generated_code="output.value"
)

# The Phoenix Client can take this query and return the dataframe.
code_gen_df = px.Client().query_spans(query, 
                                      project_name=PROJECT_NAME, 
                                      timeout=None)

code_gen_df.head()

,generated_code
context.span_id,
72238204e09cdcd1,import pandas as pd\nimport matplotlib.pyplot ...
9d70535d3b924f40,import pandas as pd\nimport matplotlib.pyplot ...
6ae259db494c3d6b,None
56cad383a70cf583,None
89c92f7d8d251db8,None


In [14]:
# code-based evaluation
def code_is_runnable(output: str) -> bool:
    """Check if the code is runnable"""
    if output:
        output = output.strip()
        output = output.replace("```python", "").replace("```", "")
    else:
        return False
    try:
        exec(output)
        return True
    except Exception as e:
        return False

In [15]:
code_gen_df["label"] = code_gen_df["generated_code"].apply(code_is_runnable).map({True: "runnable", False: "not_runnable"})
code_gen_df["score"] = code_gen_df["label"].map({"runnable": 1, "not_runnable": 0})


In [16]:
code_gen_df.head()

,generated_code,label,score
context.span_id,,,
72238204e09cdcd1,import pandas as pd\nimport matplotlib.pyplot ...,not_runnable,0
9d70535d3b924f40,import pandas as pd\nimport matplotlib.pyplot ...,not_runnable,0
6ae259db494c3d6b,None,not_runnable,0
56cad383a70cf583,None,not_runnable,0
89c92f7d8d251db8,None,not_runnable,0
